# 03. Forensic findings for the hackathon

Ноутбук проверяет конкретные расхождения между CSV, справочником тегов, ЛИМС, ПАК и формулами ВАК.

Телеметрия читается из `../data`; книги ЛИМС/ПАК/тегов читаются из `../docs` как отдельные исходные источники. Ничего не изменяется.

In [ ]:
from pathlib import Path
import sys, re
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display, Markdown

HERE = Path.cwd().resolve()
EDA_DIR = HERE if HERE.name == 'eda' else HERE / 'eda'
ROOT = EDA_DIR.parent
DATA_DIR, DOCS_DIR = ROOT / 'data', ROOT / 'docs'
ARTIFACTS = EDA_DIR / 'artifacts'
ARTIFACTS.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(EDA_DIR))
from eda_utils import load_telemetry
from forensics_utils import (parse_lims, parse_pak, constant_episodes, read_tag_reference,
    tag_mapping_audit, sulfur_matches, hydro_vak_predictions, validate_vak_against_lims)

paths = {
    'AVT telemetry': DATA_DIR / 'avt_tags.csv',
    '24-2000 telemetry': DATA_DIR / '242000_tags.csv',
    'LIMS': next(DOCS_DIR.glob('ЛИМС*.xlsx')),
    'PAK': next(DOCS_DIR.glob('Выгрузка ПАК*.xlsx')),
    'Tag reference': next(DOCS_DIR.glob('Теги_хакатон*.xlsx')),
}
display(pd.DataFrame([{'source': k, 'file': v.name, 'size_mb': v.stat().st_size/1024**2} for k,v in paths.items()]))

In [ ]:
avt = load_telemetry(paths['AVT telemetry'])
hydro = load_telemetry(paths['24-2000 telemetry'])
lims = parse_lims(paths['LIMS'])
pak = parse_pak(paths['PAK'])
mapping, pak_ref, vak_ref = read_tag_reference(paths['Tag reference'])
print('LIMS tidy:', lims.shape, '| PAK tidy:', pak.shape)

## 1. Соответствие CSV и справочника

Набор и порядок коротких имён совпадают, но это **не подтверждает масштаб и единицы**. Для 24-2000 семантика описаний часто противоречит букве тега и масштабу значений. Это делает поле description ненадёжным до ответа организаторов.

In [ ]:
telemetry = {'АВТ': avt, '24-2000': hydro}
mapping_audit = tag_mapping_audit(mapping, telemetry)
name_checks = []
for installation, frame in telemetry.items():
    csv_tags = [c for c in frame.columns if c != 'date']
    ref_tags = mapping.query('installation == @installation')['tag'].tolist()
    name_checks.append({'installation': installation, 'csv_tag_count': len(csv_tags),
                        'reference_tag_count': len(ref_tags), 'same_order': csv_tags == ref_tags,
                        'same_set': set(csv_tags) == set(ref_tags)})
display(pd.DataFrame(name_checks))
conflicts = mapping_audit.query("installation == '24-2000' and semantic_conflict")
display(conflicts)
mapping_audit.to_csv(ARTIFACTS / 'tag_mapping_forensic_audit.csv', index=False)
px.bar(mapping_audit.groupby('installation', as_index=False)['semantic_conflict'].sum(),
       x='installation', y='semantic_conflict', title='Явные конфликты description и типа тега').show()

### Тег серы

В справочнике: `24-2000:Mg.Sulfur.Q`; в выгрузке: `24-2000:Mg.Sulfur`. В обоих файлах есть только один поточный сигнал серы, поэтому alias вероятен, но не доказан. В pipeline нужен явный alias со статусом `ASSUMPTION_PENDING_CONFIRMATION`.

## 2. Ошибки единиц и заголовков ЛИМС

In [ ]:
unit_mode = (lims.groupby('quality_parameter')['unit'].agg(lambda s: s.mode().iat[0])
             .rename('unit_seen_most_often').reset_index())
unit_audit = lims[['installation','sampling_point','product','quality_parameter','unit']].drop_duplicates().merge(unit_mode)
unit_audit['unit_conflict'] = unit_audit['unit'] != unit_audit['unit_seen_most_often']
display(unit_audit.query('unit_conflict'))
unit_audit.to_csv(ARTIFACTS / 'lims_unit_audit.csv', index=False)

lims_counts = (lims.groupby(['installation','sampling_point','product','quality_parameter','unit'])
               .agg(n=('value','size'), start=('timestamp','min'), end=('timestamp','max'),
                    median=('value','median'), min=('value','min'), max=('value','max')).reset_index())
display(lims_counts)
lims_counts.to_csv(ARTIFACTS / 'lims_quality_inventory.csv', index=False)

В первом блоке АВТ единицы сдвинуты/перепутаны:

- `50%.T`: указано кг/м³, по другим точкам и смыслу должно быть °C;
- `EBP.T`: указано % об., должно быть °C;
- `D15`: указано °C, должно быть кг/м³;
- `I350`: указано °C, должно быть % об.

Кроме того, первая точка АВТ названа в ЛИМС `Дизельное топливо`, а на листе `ЛА` — `ФРАКЦ_ДИЗ`. Точку нельзя безусловно трактовать как товарный продукт.

## 3. ПАК: покрытие и frozen signals

In [ ]:
pak_summary = (pak.groupby(['tag','unit']).agg(n=('value','size'), start=('timestamp','min'), end=('timestamp','max'),
               median=('value','median'), p01=('value',lambda s:s.quantile(.01)),
               p99=('value',lambda s:s.quantile(.99)), min=('value','min'), max=('value','max'),
               unique_values=('value','nunique')).reset_index())
freezes = constant_episodes(pak, min_rows=6)
display(pak_summary, freezes.head(20))
pak_summary.to_csv(ARTIFACTS / 'pak_quality_summary.csv', index=False)
freezes.to_csv(ARTIFACTS / 'pak_frozen_episodes.csv', index=False)
px.timeline(freezes.head(20), x_start='start', x_end='end', y='tag', color='value',
            hover_data=['duration_hours','rows'], title='Самые длинные constant episodes ПАК').show()

Критическая находка: сера была ровно `7.705161` ppm с 2024-03-16 18:20 до 2024-05-02 12:10 — 6 732 точки, или 1 122 часа. Есть и другие многодневные заморозки. ПАК нельзя подавать в модель без `freshness/frozen`-флага.

## 4. Сера: ЛИМС vs ПАК

In [ ]:
sulfur_sources = lims[lims['quality_parameter'].str.contains('Sulfur')].groupby(
    ['installation','sampling_point','product','quality_parameter','unit']
).agg(n=('value','size'), start=('timestamp','min'), end=('timestamp','max'), median=('value','median'),
      p01=('value',lambda s:s.quantile(.01)), p99=('value',lambda s:s.quantile(.99)),
      min=('value','min'), max=('value','max')).reset_index()
display(sulfur_sources)

matched = sulfur_matches(lims, pak)
valid = matched.dropna(subset=['pak_ppm']).copy()
def sulfur_metrics(frame, label):
    err = frame['pak_minus_lims']
    truth, pred = frame['lims_over_10'], frame['pak_over_10']
    tp, fp = int((truth & pred).sum()), int((~truth & pred).sum())
    fn, tn = int((truth & ~pred).sum()), int((~truth & ~pred).sum())
    return {'scope': label, 'n': len(frame), 'MAE': err.abs().mean(), 'median_AE': err.abs().median(),
            'bias_PAK_minus_LIMS': err.mean(), 'spearman': frame[['lims_mg_kg','pak_ppm']].corr(method='spearman').iloc[0,1],
            'TP':tp,'FP':fp,'FN':fn,'TN':tn,'precision_over_10':tp/max(tp+fp,1),'recall_over_10':tp/max(tp+fn,1)}
metrics = pd.DataFrame([sulfur_metrics(valid,'all'), sulfur_metrics(valid.query('lims_mg_kg <= 50'),'LIMS <= 50 mg/kg')])
display(metrics)
valid.to_csv(ARTIFACTS / 'lims_pak_sulfur_matches.csv', index=False)
metrics.to_csv(ARTIFACTS / 'lims_pak_sulfur_metrics.csv', index=False)

In [ ]:
px.scatter(valid.query('lims_mg_kg <= 50'), x='lims_mg_kg', y='pak_ppm', opacity=.45,
           marginal_x='histogram', marginal_y='histogram', title='Сера: ЛИМС vs ближайший ПАК (без LIMS > 50)').show()
px.scatter(valid, x='lims_timestamp', y='pak_minus_lims', color='lims_over_10',
           title='Ошибка ПАК − ЛИМС во времени').show()
display(valid.nlargest(15, 'lims_mg_kg')[['lims_timestamp','lims_mg_kg','pak_ppm','pak_minus_lims']])

Для товарного дизеля (Гидроочистка, точка 2) ЛИМС имеет 1 462 значения, медиану 8.6 мг/кг и 15.7% значений выше 10 мг/кг. Есть явные выбросы 2 120 и 120 мг/кг.

В точках ЛИМС пороговая классификация ПАК по 10 ppm имеет recall около 26% и precision около 35%. Это не является окончательной оценкой анализатора, пока не подтверждены sampling point и смысл timestamp. Но это сильный аргумент не считать ПАК ground truth.

`Mass.Sulfur` в точке 1 имеет медиану около 0.946% масс. (около 9 460 мг/кг) и относится к другому/промежуточному продукту. Её нельзя смешивать с порогом 10 мг/кг.

## 5. Аудит формул ВАК

In [ ]:
vak_rows = []
section = None
for pair_col in (0,2,4):
    section = vak_ref.iat[0,pair_col]
    for r in range(1,len(vak_ref)):
        target, formula = vak_ref.iat[r,pair_col], vak_ref.iat[r,pair_col+1]
        if pd.notna(target):
            text = str(formula)
            vak_rows.append({'section': section, 'target': target, 'formula': text,
                'parentheses_balance': text.count('(')-text.count(')'),
                'requires_missing_pipeline_lims': 'LIMS:24-2000.Pipeline' in text,
                'has_dangling_0_00011': text.rstrip().endswith('+0.00011')})
vak_audit = pd.DataFrame(vak_rows)
display(vak_audit)
vak_audit.to_csv(ARTIFACTS / 'vak_formula_text_audit.csv', index=False)

pred = hydro_vak_predictions(hydro)
validation = validate_vak_against_lims(pred, lims)
display(validation.sort_values('median_absolute_error'))
validation.to_csv(ARTIFACTS / 'vak_hydro_validation.csv', index=False)
px.bar(validation.sort_values('median_absolute_error'), x='target', y='median_absolute_error',
       color='spearman', title='ВАК 24-2000: median absolute error на ЛИМС').show()

In [ ]:
vak_runtime_risks = pd.DataFrame([
 {'risk':'T90 scale/reference error','evidence':'first prediction ≈166286; median ≈202711 vs LIMS median 331 °C'},
 {'risk':'T50 scale/reference error','evidence':'matched prediction median ≈392 vs LIMS median 274 °C'},
 {'risk':'CloudPoint formula truncated or wrong tag','evidence':'formula ends with bare +0.00011; median ≈10929 vs LIMS -4 °C'},
 {'risk':'CFPP wrong scale/mapping','evidence':'matched prediction median ≈10.35 vs LIMS -6 °C'},
 {'risk':'Division by zero in hydro T90','evidence':f"F26 equals zero in {int(hydro['F26'].eq(0).sum())} rows"},
 {'risk':'Division by zero in AVT formulas','evidence':f"F57 equals zero in {int(avt['F57'].eq(0).sum())} rows"},
 {'risk':'Missing Pipeline LIMS','evidence':'24-2000 D15 and T95 formulas require unavailable LIMS:24-2000.Pipeline variables'},
 {'risk':'AVT CFPP syntax ambiguity','evidence':'formula has one extra closing parenthesis; F65/F32+F30 differs materially from F65/(F32+F30)'},
])
display(vak_runtime_risks)
vak_runtime_risks.to_csv(ARTIFACTS / 'vak_runtime_risks.csv', index=False)

`IBP` — самый правдоподобный из воспроизводимых ВАК: median AE около 4.8°C, Spearman около 0.54. `I250` имеет median AE около 3.8 п.п., но слабую ранговую связь. Остальные формулы нельзя использовать без исправлений, масштабирования и validity masks.

## 6. Что использовать на хакатоне

1. **Ground truth:** `LIMS / Гидроочистка / точка 2 / Mg.Sulfur`; отдельно показать строки 2 120/120 мг/кг как data-quality review.
2. **ПАК:** использовать как частый, но менее доверенный сигнал. Обязательны `age`, `time_since_change`, frozen flag и fallback.
3. **Теги:** не присваивать F25 смысл температуры вспышки, пока mapping не исправлен; медиана F25 около 13 091.
4. **ВАК:** показать как baseline only. IBP можно сохранить как benchmark; остальные — через gates.
5. **Модель:** только temporal split/walk-forward. Признаки X(t−lag), target LIMS(t), а при online replay — availability time анализа, если оно будет уточнено.
6. **Рекомендации:** без симулятора называть их model-based what-if, а не доказанным эффектом.
7. **Блендинг:** не обучать реальную blending model без рецептур/расходов; показать отдельный сценарий с явными допущениями.